# Part B — Per-Axis Torque Box

**ES4G4 Assignment 1 (Python rebuild)** · The Part B design with the torque limit read as an
independent per-axis box, $|\tau_i| \le 0.25$ N·m, instead of the shared budget
$\|\boldsymbol\tau\|_2 \le 0.25$ N·m of `part_b.ipynb`. Same plant, same weights, same
horizons — the constraint set is the only independent variable. Standalone: the design is
rebuilt from `quadrotor.py` and `mpc.py`.

In [ ]:
# The B-1 design, rebuilt from the shared modules and instantiated twice: once with
# the shared l2 budget of part_b.ipynb, once with the per-axis box. Both controllers get
# the SAME (Q, R, RD, P) from one weights_for call, so nothing but the constraint differs.
import time

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize

from quadrotor import linearised_matrices, PARAMS, TAU_MAX, DEG
from mpc import (discretise, weights_for, MPC, simulate, step_metrics, impulse_of,
                 show, P_H, C_H)

np.set_printoptions(precision=4, suppress=True, linewidth=120)

Ts = 2e-3                                  # §2  sample time
ZETA, WN = 1.0, 500.0                      # §6/§7  damping ratio, target bandwidth
TAU_D, T_D = 0.1, 5e-3                     # §2  disturbance pulse amplitude, duration
Ix = PARAMS["Ix"]

plant = discretise(*linearised_matrices(), Ts)
n, m = plant.Bd.shape
W = weights_for(WN, plant, ZETA)           # (Q, R, RD, P) — shared by both controllers
CTRL = {"l2": MPC(plant, *W), "box": MPC(plant, *W, budget="box")}

# the B-2 scenario, unchanged
ref_b2 = np.zeros(n); ref_b2[0], ref_b2[1] = 10 * DEG, -10 * DEG
x0_b2 = np.zeros(n); x0_b2[0], x0_b2[1] = 5 * DEG, -5 * DEG
ref_b2_of_t = lambda tk: ref_b2 if tk >= 10.0 else np.zeros(n)

# Per-axis authority on a simultaneous roll+pitch slew — the one number that propagates.
TAU_AX = {"l2": TAU_MAX / np.sqrt(2), "box": TAU_MAX}
t_bb = lambda I_, g: 2 * np.sqrt(I_ * (10 * DEG) / TAU_AX[g])    # §5 bang-bang slew bound

print(f"per-axis authority, roll+pitch together:  l2 {TAU_AX['l2']:.4f} N.m   "
      f"box {TAU_AX['box']:.4f} N.m   ({TAU_AX['box'] / TAU_AX['l2']:.3f}x)")
for g in CTRL:
    print(f"  {g:>3}: t_bb = {t_bb(Ix, g) * 1e3:4.1f} ms nominal, "
          f"{t_bb(1.55 * Ix, g) * 1e3:4.1f} ms at +55% inertia -> c = {C_H} "
          f"({C_H * Ts * 1e3:.0f} ms) spans {C_H * Ts / t_bb(1.55 * Ix, g):.2f}x "
          f"that bound")
print(f"\nhorizons held at p = {P_H}, c = {C_H} for both geometries: the constrained "
      f"interval shrank under\nthe box, so the §5 sizing rule is satisfied with more "
      f"margin, not less. Holding them fixed\nkeeps this a controlled comparison.")
from pathlib import Path                  # report figures are exported
FIG = Path("figures"); FIG.mkdir(exist_ok=True)

## What changes, and what cannot

| Quantity | Under the box | Why |
|---|---|---|
| $T_s$, $\zeta$, $\omega_n$, $q_1$, $q_2$, $r$, $\mathbf{R}_\Delta$, $\mathbf{P}$ | unchanged | `weights_for` computes $r = q_1/(I_x^2\omega_n^4)$ — no $\tau_{max}$ appears in it, so the weights cannot depend on the constraint. $\mathbf{P}$ solves the *unconstrained* DARE. |
| Problem class | SOCP → **QP** | $\|u\|_2 \le \tau_{max}$ is a cone; $|u_i| \le \tau_{max}$ is linear. Solver moves Clarabel → OSQP automatically. |
| Per-axis authority, roll+pitch together | $0.1768 \to 0.250$ N·m | $\mathbf{1.414\times}$. Every downstream difference traces to this line. |
| Horizons $p, c$ | held at $25, 10$ | The constrained interval *shrinks*, so the existing horizons carry more margin. Held fixed deliberately. |

Geometrically the ball is $52\%$ of the box by volume and $70.7\%$ of its per-axis reach on a
two-axis command. Neither is "the correct reading": on a real vehicle three torques come from
four rotor thrusts through a mixer, so the admissible set is a polytope, and how $\tau_{max}$
maps to per-motor thrust — which the brief does not specify — decides where that polytope sits
relative to either idealisation. The question here is narrower and answerable: **what did the
$\ell_2$ conservatism cost, and did it cost anything on the specification that binds?**

### Predictions, stated before the run

1. **B-3 is unchanged, to solver tolerance.** `part_b.ipynb` measures peak
   $\|\boldsymbol\tau\|_2 = 0.2016$ N·m during disturbance rejection — $81\%$ of budget, so
   the ball constraint is *never active*. The ball is contained in the box, so an inactive
   ball implies an inactive box and both programmes return the same unconstrained optimum.
   Not bit-for-bit: Clarabel gives way to OSQP, a first-order method. Checked as a numerical
   difference below, not asserted as exact.
2. **B-2 gets faster by $\approx 1.19\times$.** Rise scales as $1/\sqrt{\tau_{axis}}$, so
   $\sqrt{0.1768/0.25} = 0.841$: $10$ ms $\to \approx 8.4$ ms. That is under two samples at
   $T_s = 2$ ms, so the rounded metric is coarse — the trajectories are the evidence. Per-axis
   torque should reach $0.250$ on roll *and* pitch simultaneously, which the ball forbids, and
   $\|\boldsymbol\tau\|_2$ should climb to $0.354$.
3. **Overshoot stays at zero.** $\zeta = 1$ is a property of the weights, not the constraint.
4. **B-4's direction is genuinely unknown.** The B-4 failure is model mismatch — a heavier
   plant defeats a brake planned on nominal inertia — not saturation, so extra authority need
   not help. But the $\ell_2$ ensemble saturates for $16$ ms at the worst corner, so if the
   brake was *also* torque-limited there, the box will help. Both hypotheses are live; the run
   decides.

In [ ]:
# B-2 under both geometries. step_metrics takes the same budget as the controller,
# so peak_u and the saturation columns are measured against the constraint that applies.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]      # x/roll, y/pitch, z/yaw — fixed order
INK, MUTED, GRID_C, CRIT = "#52514e", "#898781", "#e1e0d9", "#d03b3b"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "font.size": 11, "axes.titlesize": 11, "axes.labelsize": 11,
    "legend.fontsize": 9.5, "xtick.labelsize": 9.5, "ytick.labelsize": 9.5,
    "axes.grid": True, "grid.color": GRID_C, "grid.linewidth": 0.8,
    "axes.edgecolor": "#c3c2b7", "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "xtick.labelcolor": INK, "ytick.labelcolor": INK,
    "lines.linewidth": 1.8, "legend.frameon": False,
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
})

B2 = {}
rows = {}
for g, ctrl in CTRL.items():
    t2, X2, U2, _ = simulate(ctrl, plant, x0_b2, ref_b2_of_t, T_end=15.0)
    B2[g] = (t2, X2, U2)
    for nm, tgt, ax_ in (("roll", 10 * DEG, 0), ("pitch", -10 * DEG, 1)):
        rows[f"{g} step {nm}"] = step_metrics(X2, U2, Ts, 10.0, tgt, axis=ax_,
                                              budget=g)
show(rows, label="B-2 step at t = 10 s")

for g in CTRL:
    U2 = B2[g][2]
    print(f"{g:>3}: peak per-axis |tau| = {np.abs(U2).max(0).round(4)} N.m, "
          f"peak ||tau||_2 = {np.linalg.norm(U2, axis=1).max():.4f} N.m")
# the active constraint is met to solver tolerance, not exactly: OSQP is first-order
resid = np.abs(B2["box"][2]).max() - TAU_MAX
print(f"box constraint residual = {resid:+.2e} N.m (OSQP tolerance, not a violation)")
assert resid < 1e-4, "box budget violated"
r_rise = rows["box step roll"]["rise_ms"] / rows["l2 step roll"]["rise_ms"]
print(f"\nFINDING  the box speeds the step up: rise x{r_rise:.2f} "
      f"(predicted x{np.sqrt(TAU_AX['l2'] / TAU_AX['box']):.2f}), "
      f"overshoot {rows['box step roll']['overshoot_pc']:.2f}% vs "
      f"{rows['l2 step roll']['overshoot_pc']:.2f}%")

# --- figure: angles and torque, zoomed on the step, l2 beside box ------------
K0, K1 = int(round(10.0 / Ts)), int(round(10.06 / Ts))
tms = (np.arange(K0, K1) - K0) * Ts * 1e3
fig, ax = plt.subplots(2, 2, figsize=(12.5, 7.6), sharey="row",
                       gridspec_kw=dict(hspace=0.30, wspace=0.16))
for col, g in enumerate(CTRL):
    _, X2, U2 = B2[g]
    a = ax[0, col]
    for s, lab in enumerate([r"$\phi$ roll", r"$\theta$ pitch", r"$\psi$ yaw"]):
        a.plot(tms, X2[K0:K1, s] / DEG, color=SERIES[s], label=lab)
    for tgt in (10.0, -10.0):
        a.axhline(tgt, color=MUTED, ls="--", lw=1.2)
    a.set(title=f"{g}   attitude", ylabel="angle (deg)")
    if col == 0:
        a.legend(loc="center right")

    a = ax[1, col]
    for s, lab in enumerate([r"$\tau_x$", r"$\tau_y$", r"$\tau_z$"]):
        a.plot(tms, U2[K0:K1, s], color=SERIES[s], drawstyle="steps-post", label=lab)
    a.plot(tms, np.linalg.norm(U2[K0:K1], axis=1), color=INK, lw=1.3, alpha=0.7,
           drawstyle="steps-post", label=r"$\|\tau\|_2$")
    for lim in (TAU_MAX, -TAU_MAX):
        a.axhline(lim, color=CRIT, ls="--", lw=1.4)
    a.set(title=f"{g}   torque  (limit lines at $\\pm${TAU_MAX})",
          xlabel="time after step (ms)", ylabel=r"torque (N$\cdot$m)")
    if col == 0:
        a.legend(loc="center right", ncol=2)
fig.suptitle("B-2 — the box lets roll and pitch each take the full 0.25 N·m, which the "
             "shared budget forbids", color=INK, fontsize=12, y=0.955)
fig.savefig(FIG / "box_b2.png")
plt.show()

In [ ]:
# B-3 under both geometries. The prediction is that nothing happens: the constraint
# is inactive during rejection, and the ball sits inside the box.
TAU_DIST = np.array([0.1, -0.1, 0.05])
impulse = impulse_of(TAU_DIST, 1.0, Ts, T_D)


def reject_metrics(X, U, axis, g, t0=1.0, band=0.02):
    """Excursion and recovery for the disturbance transient starting at t0."""
    k0 = int(round(t0 / Ts))
    y = np.abs(X[k0:, axis])
    outside = y > band * y.max()
    un = (np.linalg.norm(U[k0:], axis=1) if g == "l2" else np.abs(U[k0:]).max(axis=1))
    return dict(peak_deg=y.max() / DEG, t_peak_ms=float(y.argmax()) * Ts * 1e3,
                recover_ms=(np.flatnonzero(outside)[-1] + 1) * Ts * 1e3
                if outside.any() else 0.0, peak_u=float(un.max()))


B3, rows3 = {}, {}
for g, ctrl in CTRL.items():
    _, X3, U3, _ = simulate(ctrl, plant, np.zeros(n), lambda _t: np.zeros(n),
                            T_end=4.0, dist_of_t=impulse)
    B3[g] = (X3, U3)
    for a_, nm in enumerate(("roll ", "pitch", "yaw  ")):
        rows3[f"{g} {nm}"] = reject_metrics(X3, U3, a_, g)
show(rows3, label="B-3 impulse at t = 1 s")

dU = np.abs(B3["box"][1] - B3["l2"][1]).max()
dX = np.abs(B3["box"][0] - B3["l2"][0]).max() / DEG
print(f"\nFINDING  the two geometries give the same B-3 response: max|dtau| = {dU:.2e} "
      f"N.m, max|dangle| = {dX:.2e} deg")
print(f"         peak ||tau||_2 = {np.linalg.norm(B3['l2'][1], axis=1).max():.4f} N.m = "
      f"{100 * np.linalg.norm(B3['l2'][1], axis=1).max() / TAU_MAX:.0f}% of budget "
      f"-> the constraint is never active, so the reading cannot matter")
assert dU < 5e-3, "the two solutions differ by more than solver tolerance"

# --- figure: the two responses overlaid; they should be indistinguishable ----
K1 = int(round(1.08 / Ts))
K0 = int(round(1.0 / Ts))
tms3 = (np.arange(K0, K1) - K0) * Ts * 1e3
fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.4), gridspec_kw=dict(wspace=0.22))
for s, lab in enumerate([r"$\phi$ roll", r"$\theta$ pitch", r"$\psi$ yaw"]):
    ax[0].plot(tms3, B3["l2"][0][K0:K1, s] / DEG, color=SERIES[s], label=f"{lab}  l2")
    ax[0].plot(tms3, B3["box"][0][K0:K1, s] / DEG, color=INK, lw=1.0, ls="--",
               label="box" if s == 0 else None)
for lim in (1.0, -1.0):
    ax[0].axhline(lim, color=CRIT, ls=":", lw=1.4)
ax[0].annotate(r"$\pm1°$ deviation budget", (tms3[-1], 1.0), color=CRIT, fontsize=9,
               ha="right", va="bottom", xytext=(0, 3), textcoords="offset points")
ax[0].set(xlabel="time after impulse (ms)", ylabel="angle (deg)",
          title="attitude — box (ink, dashed) lies on top of l2")
ax[0].legend(loc="center right", ncol=2)

for s, lab in enumerate([r"$\tau_x$", r"$\tau_y$", r"$\tau_z$"]):
    ax[1].plot(tms3, B3["l2"][1][K0:K1, s], color=SERIES[s], drawstyle="steps-post",
               label=f"{lab}  l2")
    ax[1].plot(tms3, B3["box"][1][K0:K1, s], color=INK, lw=1.0, ls="--",
               drawstyle="steps-post", label="box" if s == 0 else None)
ax[1].axhline(TAU_MAX, color=CRIT, ls="--", lw=1.4)
ax[1].annotate(r"$\tau_{max}$ — never reached", (tms3[-1], TAU_MAX), color=CRIT,
               fontsize=9, ha="right", va="bottom", xytext=(0, 3),
               textcoords="offset points")
ax[1].set(xlabel="time after impulse (ms)", ylabel=r"torque (N$\cdot$m)",
          title="torque — the constraint is never active", ylim=(-0.28, 0.28))
ax[1].legend(loc="lower right", ncol=2)
fig.suptitle("B-3 — identical under both readings, because the binding specification "
             "never touches the constraint", color=INK, fontsize=12, y=1.0)
fig.savefig(FIG / "box_b3.png")
plt.show()

In [ ]:
# B-4 under the box: the same 15 seeded plants as part_b4.ipynb (same seed, so the
# factors match run for run), driven by the box controller with nominal (Ad, Bd).
KEYS = ("Ix", "Iy", "Iz", "dx", "dy", "dz")
F_I, F_D, N_SAMPLES = 0.55, 0.75, 15

rng = np.random.default_rng(4)                          # same seed as part_b4.ipynb
FAC = np.hstack([rng.uniform(1 - F_I, 1 + F_I, (N_SAMPLES, 3)),
                 rng.uniform(1 - F_D, 1 + F_D, (N_SAMPLES, 3))])
FAC = np.vstack([FAC, [1 + F_I] * 3 + [1 - F_D] * 3])   # worst corner, not a sample
LABELS = [f"sample {i + 1:02d}" for i in range(N_SAMPLES)] + ["worst corner"]

XU, rows4 = [], {}
t_wall = time.perf_counter()
for i, (lab, fac) in enumerate(zip(LABELS, FAC)):
    p_s = {**PARAMS, **{k: f * PARAMS[k] for k, f in zip(KEYS, fac)}}
    _, Xs, Us, _ = simulate(CTRL["box"], discretise(*linearised_matrices(p_s), Ts),
                            x0_b2, ref_b2_of_t, T_end=15.0)
    mr = step_metrics(Xs, Us, Ts, 10.0, 10 * DEG, axis=0, budget="box")
    mp = step_metrics(Xs, Us, Ts, 10.0, -10 * DEG, axis=1, budget="box")
    XU.append((Xs, Us))
    rows4[lab] = dict(f_Ix=fac[0], f_Iy=fac[1], f_dx=fac[3], f_dy=fac[4],
                      os_roll=mr["overshoot_pc"], os_pitch=mp["overshoot_pc"],
                      rise_ms=max(mr["rise_ms"], mp["rise_ms"]),
                      settle_ms=max(mr["settle_ms"], mp["settle_ms"]))
    el, left = time.perf_counter() - t_wall, len(LABELS) - 1 - i
    print(f"\r[{'#' * (i + 1)}{'.' * left}] {i + 1}/{len(LABELS)}  {el:3.0f} s elapsed"
          f", ~{el / (i + 1) * left:3.0f} s left  ({lab})   ", end="", flush=True)
print(f"\n\n{len(LABELS)} runs x {len(XU[0][1])} solves in {el:.0f} s wall clock\n")
show(rows4, label="run (f = factor)")

OS = np.array([[rows4[l]["os_roll"], rows4[l]["os_pitch"]] for l in LABELS])
fI, fd, s = FAC[:, [0, 1]], FAC[:, [3, 4]], slice(None, N_SAMPLES)
r_I = np.corrcoef(fI[s].ravel(), OS[s].ravel())[0, 1]
r_d = np.corrcoef(fd[s].ravel(), OS[s].ravel())[0, 1]
n_breach = (OS[s].max(1) > 5).sum()
Ua = np.array([u for _, u in XU])

# the l2 ensemble of part_b4.ipynb, for comparison
L2 = dict(n_breach=5, worst_sample=8.036, corner=11.08, r_I=0.876, r_d=-0.124)
print(f"\nFINDING  {n_breach} of {N_SAMPLES} samples breach the 5% overshoot spec under "
      f"the box, against {L2['n_breach']} of {N_SAMPLES} under l2 (part_b4.ipynb)")
print(f"FINDING  worst sample {OS[s].max():.2f}% vs {L2['worst_sample']:.2f}% (l2); "
      f"worst corner {OS[-1, 0]:.2f}% vs {L2['corner']:.2f}% (l2)")
print(f"FINDING  the mechanism is unchanged: inertia drives overshoot (r = {r_I:+.2f}, "
      f"l2 {L2['r_I']:+.2f}), damping does not (r = {r_d:+.2f}, l2 {L2['r_d']:+.2f})")
print(f"FINDING  rise <= {max(rows4[l]['rise_ms'] for l in LABELS):.0f} ms on every run; "
      f"peak per-axis |tau| = {np.abs(Ua).max():.4f} N.m (limit {TAU_MAX})")

In [ ]:
# B-4 figure, same construction as part_b4.ipynb so the two sit side by side:
# colour carries the inertia factor of the axis plotted, the corner is ink, and the
# panel titles carry the finding.
RAMP = LinearSegmentedColormap.from_list("inertia", ["#5b96d8", "#0c2f5e"])
NORM = Normalize(1 - F_I, 1 + F_I)
K0, K1 = int(round(10.0 / Ts)), int(round(10.06 / Ts))
tms4 = (np.arange(K0, K1) - K0) * Ts * 1e3
CORNER = N_SAMPLES

fig, ax = plt.subplots(2, 2, figsize=(12.5, 8.4),
                       gridspec_kw=dict(hspace=0.34, wspace=0.22))
for col, (axis, tgt, title) in enumerate(
        [(0, 10.0, r"roll $\phi$: step to $+10°$"),
         (1, -10.0, r"pitch $\theta$: step to $-10°$")]):
    a = ax[0, col]
    for i, (Xs, _) in enumerate(XU):
        y = Xs[K0:K1, axis] / DEG
        if i == CORNER:
            a.plot(tms4, y, color=INK, lw=2.2, ls="--", zorder=3)
        else:
            a.plot(tms4, y, color=RAMP(NORM(FAC[i, axis])), lw=1.4, alpha=0.9)
    a.axhline(tgt, color=MUTED, ls="--", lw=1.2)
    a.axhline(tgt * 1.05, color=CRIT, ls=":", lw=1.4)
    a.annotate("5% overshoot limit", (tms4[-1], tgt * 1.05), color=CRIT, fontsize=9,
               ha="right", va="bottom" if tgt > 0 else "top",
               xytext=(0, 3 * np.sign(tgt)), textcoords="offset points")
    a.set(xlabel="time after step (ms)", ylabel="angle (deg)", title=title)

dot = dict(s=48, cmap=RAMP, norm=NORM, edgecolor="#fcfcfb", linewidths=0.8, zorder=3)
for col, (xv, xlabel, title) in enumerate(
        [(fI, "inertia factor $f_I$ of that axis",
          f"inertia drives overshoot:  $r = {r_I:+.2f}$"),
         (fd, "damping factor $f_d$ of that axis",
          f"damping does not:  $r = {r_d:+.2f}$")]):
    a = ax[1, col]
    a.scatter(xv[s].ravel(), OS[s].ravel(), c=fI[s].ravel(), **dot)
    a.scatter(xv[CORNER, 0], OS[CORNER, 0], marker="D", s=70, color=INK, zorder=4)
    side = -1 if col == 0 else 1        # keep the label inside the axes either side
    a.annotate("worst corner", (xv[CORNER, 0], OS[CORNER, 0]), color=INK, fontsize=9,
               ha="right" if col == 0 else "left", va="center",
               xytext=(12 * side, 0), textcoords="offset points")
    a.axhline(5.0, color=CRIT, ls=":", lw=1.4)
    a.set(xlabel=xlabel, ylabel="overshoot (%)", title=title, ylim=(-0.6, 12.0))

cb = fig.colorbar(plt.cm.ScalarMappable(norm=NORM, cmap=RAMP), ax=ax.ravel().tolist(),
                  location="right", fraction=0.025, pad=0.015)
cb.set_label("inertia factor $f_I$ of the plotted axis", color=INK)
cb.ax.tick_params(color=MUTED, labelcolor=INK)
cb.outline.set_edgecolor("#c3c2b7")
fig.suptitle(f"B-4 under the box — {n_breach} of {N_SAMPLES} plants breach the 5% "
             f"overshoot spec (l2: {L2['n_breach']} of {N_SAMPLES})",
             color=INK, fontsize=12, y=0.955)
fig.savefig(FIG / "box_b4.png")
plt.show()

## Results — what the $\ell_2$ conservatism cost

> **On the specification that binds, it cost nothing.** B-3 is the same response under both
> readings — $\max|\Delta\tau| = 2.5\times10^{-9}$ N·m, $\max|\Delta\text{angle}| =
> 5.3\times10^{-6}$ deg. Peak demand is $0.2016$ N·m, $81\%$ of budget, so the constraint is
> never active and the two programmes solve the same unconstrained problem.
>
> **On tracking, it cost about $20\%$ of step speed.** Rise $10 \to 8$ ms ($\times0.80$), with
> the constrained interval halved from $8$ ms in two episodes to $4$ ms in one.
>
> **On robustness, it cost severity but not incidence.** The same $5$ of $15$ plants breach
> the $5\%$ spec, but the worst corner falls from $11.08\%$ to $7.43\%$ — a third lower.
>
> **The mechanism is untouched.** Inertia still drives overshoot ($r = +0.87$ against $+0.88$),
> damping still does not ($r = -0.08$ against $-0.12$).

**All four predictions hold.** B-3 unchanged; B-2 faster by $\times0.80$ against a predicted
$\times0.84$ — the gap is quantisation, since rise time resolves only to $T_s = 2$ ms and the
prediction fell between two samples; overshoot zero throughout; and B-4 resolved by
measurement rather than assertion, below.

**One column that looks like a difference and is not.** `peak_u` reads $0.2016$ under
$\ell_2$ and $0.1362$ under the box in the B-3 table. Both describe the *same* torque
trajectory, measured against the constraint that applies to it: $\|\boldsymbol\tau\|_2$ in one
case, $\max_i|\tau_i|$ in the other. This is why `step_metrics` takes the geometry as an
argument — measuring a box-constrained run with the $\ell_2$ test would report saturation
almost everywhere, since $\|\boldsymbol\tau\|_2$ reaches $0.354$ with no axis at its limit.

**B-4: both hypotheses were right, in different regions.** The extra authority helps exactly
where the brake was torque-limited and nowhere else. At the worst corner — where the $\ell_2$
run saturated for $16$ ms — overshoot falls by a third. Near the threshold, where saturation
is brief and the failure is purely that the controller mispredicts a heavier plant, the box
does almost nothing: sample 01 improves $6.80 \to 6.59\%$ while sample 10 *worsens*
$6.53 \to 6.81\%$, both within the noise of a different active-set trajectory. So the box
compresses the tail without moving the threshold, and the breach count is unchanged. **Extra
control authority cannot fix a wrong prediction** — which is the same conclusion B-4 reached
under $\ell_2$, now established by intervention rather than by argument.

**Horizons.** The bang-bang slew bound shortens from $15.4$ to $12.9$ ms nominal and $19.2$ to
$16.1$ ms at $+55\%$ inertia, so $c = 10$ ($20$ ms) goes from spanning $1.04\times$ that bound
to $1.24\times$. §5's sizing rule is satisfied with more margin under the box, not less, which
is why holding $p = 25$, $c = 10$ fixed is safe as well as necessary for a controlled
comparison.

**Which reading to use.** For this brief it does not matter much: the two agree exactly on the
binding specification and differ by one sample of rise time on the other. The box is the more
literal reading of $|\tau| \le 0.25$ and gives a cheaper QP (OSQP rather than Clarabel), so it
is the better default for Parts C and D. The $\ell_2$ ball remains the more defensible model
if the four rotors share a thrust envelope — but that is an assumption about hardware the
brief does not describe, and this notebook shows the choice costs nothing where it counts.